In [ ]:

import numpy as np
import os
import pandas as pd
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
from core.Log import *
from core.plots import *
from core.modelUtils import *
from core.CNNmodel import *
from core.CardiacCTdataset import DataLoaderFactory
#from core.preprocessing import *
from core.globals import *
#from core.globals import PARAM_GRID
import logging
from core.CVsplits import *
from tqdm.notebook import tqdm
root_logger()
folds_logger()
setup_loggers()
logger = logging.getLogger('root')
fold_log = logging.getLogger('folds')


shape=(64, 64, 64)
pools = ["holdout", "main"]




main_dataset=load_dataset(pool="main")
all_folds_data=get_fold_stats()
DL = DataLoaderFactory(main_dataset, all_folds_data)
OUTER_FOLDS = 5; INNER_FOLDS = 3

final_experiment = load_from_json(filename="training/parameter_grid.json")
final_experiment_df = pd.DataFrame(final_experiment)
final_experiment_df


C:\Users\sulei\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\requests\__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


Loaded training/parameter_grid.json.


,ExpID,Fold,HPset,LR,WD,DR,TH,epochs,trained,evaluated
0,1,0,6,0.0008,0.000100,0.2,0.5,50,True,False
1,2,1,6,0.0008,0.000100,0.2,0.5,50,False,False
2,3,2,6,0.0008,0.000100,0.2,0.5,50,False,False
3,4,3,6,0.0008,0.000100,0.2,0.5,50,False,False
4,5,0,9,0.0008,0.000001,0.2,0.5,50,False,False
5,6,1,9,0.0008,0.000001,0.2,0.5,50,False,False
6,7,2,9,0.0008,0.000001,0.2,0.5,50,False,False
7,8,3,9,0.0008,0.000001,0.2,0.5,50,False,False


In [2]:
pbar_experiments = tqdm(final_experiment, desc=f"Final Model Experiments", position=0, leave=True)
for experiment in pbar_experiments:
	fold = experiment["Fold"]
	trained = experiment["trained"]
	HPset = experiment["HPset"]
	if trained: continue
	train_loader, val_loader, test_loader = DL.create_outer_loaders(fold)
	DR = experiment['DR']
	model = MultiViewCNN(DR)
	model, best_model_state = TRAIN_MODEL(model, train_loader, val_loader, experiment)
	torch.save(best_model_state, f"pth_models/fold_{fold}_HPset_{HPset}.pth")
	experiment["trained"] = True
	save_to_json(final_experiment, filename="training/parameter_grid.json")



Final Model Experiments:   0%|          | 0/8 [00:00<?, ?it/s]

	↳ Experiment 2 | Training model... :   0%|          | 0/50 [00:00<?, ?it/s]

	↳ Experiment 3 | Training model... :   0%|          | 0/50 [00:00<?, ?it/s]

	↳ Experiment 4 | Training model... :   0%|          | 0/50 [00:00<?, ?it/s]

	↳ Experiment 5 | Training model... :   0%|          | 0/50 [00:00<?, ?it/s]

	↳ Experiment 6 | Training model... :   0%|          | 0/50 [00:00<?, ?it/s]

	↳ Experiment 7 | Training model... :   0%|          | 0/50 [00:00<?, ?it/s]

	↳ Experiment 8 | Training model... :   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
pbar_experiments = tqdm(final_experiment, desc=f"Final Model Experiments", position=0, leave=True)
for experiment in pbar_experiments:
	fold = experiment["Fold"]
	evaluated = experiment["evaluated"]
	HPset = experiment["HPset"]
	if evaluated: continue
	_, _, test_loader = DL.create_outer_loaders(fold)
	DR = experiment['DR']
	model = MultiViewCNN(DR)
	state_dict = torch.load(f"pth_models/fold_{fold}_HPset_{HPset}.pth")
	model.load_state_dict(state_dict)
	final_loss, all_probabilities, all_labels, all_predictions = EVALUATE_MODEL(model, test_loader, experiment)
	experiment["evaluated"] = True
	save_to_json(final_experiment, filename="training/parameter_grid.json")
